<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: When units affect each other

**[When units affect each other](https://learning.nextia-ai.com/courses/experiments/m04/when-units-affect-each-other/)** · Experiments and Causal Impact: Did Our Change Work? · Module 4, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** simulate a test where A and B share one stock, see why the usual comparison is too large, and check a switchback test (one version for everyone, switched by day) against the truth.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | the two worlds of the simulator, from [Causes and comparisons](https://learning.nextia-ai.com/courses/experiments/m01/causes-and-comparisons/); the interval, from [The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m04/when-units-affect-each-other/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M04-L04-when-units-affect-each-other/when-units-affect-each-other-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: the project folder

This notebook makes its own data with small simulations, so it downloads nothing. Run the next cell. It makes a folder `larkfield-experiments` and works in it.

You should see `Ready: 0 files in data/.`

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One day at a shop with one best seller

Each day, 1,000 customers visit in a random order. Each one wants the best seller with a chance of 10%, or 13% with a "popular now" badge (+30%). The shop has 100 units a day. The first customers who want it get it; after that, it is sold out.

> **Predict.** If half the customers see the badge, will the badge group get more than half of the 100 units?

In [ ]:
def shop_day(seed, treated, n=1000, stock=100, base=0.10, lift=0.30):
    """One day: who wants the best seller, and who gets it (first come, first served).
    The same seed gives the same customers and the same random numbers in every world."""
    u = np.random.default_rng(seed).random(n)
    want = u < base * np.where(treated, 1 + lift, 1.0)
    return want & (np.cumsum(want) <= stock)

coin = np.random.default_rng(1).random(1000) < 0.5        # the badge for half the customers
got = shop_day(0, coin)
print(f"units sold: {got.sum()}; to the badge group: {got[coin].sum()}, to the others: {got[~coin].sum()}")
print(f"order rate: badge {got[coin].mean():.3f}, no badge {got[~coin].mean():.3f}")

> **Check.** You should see `units sold: 99; to the badge group: 59, to the others: 40`.

The badge group wants the product more often, so it takes more of the stock. On days when the stock runs out, the other group finds the shelf empty. The test compares the two groups, but part of B's gain is A's loss.

## 2. Two worlds: the true effect of a launch

The real question is: badge for everyone against badge for nobody. The simulator can run both worlds on the same customers.

In [ ]:
def test_by_customer(seed, days=28):
    rng = np.random.default_rng(seed)
    seeds = rng.integers(0, 2**32, days)
    a, b, nobody, everyone = [], [], [], []
    for s in seeds:
        coin = np.random.default_rng(s + 1).random(1000) < 0.5
        got = shop_day(s, coin)
        a.append(got[~coin].mean()); b.append(got[coin].mean())
        nobody.append(shop_day(s, np.zeros(1000, bool)).mean())
        everyone.append(shop_day(s, np.ones(1000, bool)).mean())
    return np.mean(b) - np.mean(a), np.mean(everyone) - np.mean(nobody)

test, truth = test_by_customer(0)
print(f"test, B - A: {test:+.4f} orders per visit; true effect of a launch: {truth:+.4f}")

> **Check.** You should see `test, B - A: +0.0311 orders per visit; true effect of a launch: +0.0041`.

The test says the badge adds about 3.1 points to the order rate. A launch adds about 0.4: the stock is the limit, so the badge mostly moves who gets the units. The two groups affect each other: this is **interference**.

## 3. A switchback test: one version per day

A switchback test gives everyone the same version on the same day and switches between days. A and B never share a day's stock. The unit is now a day, so we have only 28 units.

In [ ]:
def switchback(seed, days=28):
    rng = np.random.default_rng(seed)
    seeds = rng.integers(0, 2**32, days)
    day_b = rng.random(days) < 0.5                       # the coin flip per day
    rate = np.array([shop_day(s, np.full(1000, bd)).mean() for s, bd in zip(seeds, day_b)])
    nobody = np.mean([shop_day(s, np.zeros(1000, bool)).mean() for s in seeds])
    everyone = np.mean([shop_day(s, np.ones(1000, bool)).mean() for s in seeds])
    r = stats.ttest_ind(rate[day_b], rate[~day_b], equal_var=False)
    ci = r.confidence_interval()
    return rate[day_b].mean() - rate[~day_b].mean(), ci.low, ci.high, everyone - nobody

est, low, high, truth = switchback(0)
print(f"switchback: {est:+.4f} (95% interval {low:+.4f} to {high:+.4f}); truth {truth:+.4f}")

> **Check.** You should see `switchback: +0.0029 (95% interval +0.0002 to +0.0056); truth +0.0041`.

The switchback estimate is close to the truth. Its interval is wide compared with the effect: 28 days are few units.

## 4. Your turn: coverage over 100 seeds

> **Your turn.** Run `switchback(seed)` for seeds 0 to 99. Put in `coverage` the share of seeds whose interval contains that seed's truth, rounded to 2 decimals.

In [ ]:
runs = [switchback(s) for s in range(100)]
coverage = round(float(np.mean([low <= truth <= high for est, low, high, truth in runs])), 2)
print(coverage)

In [ ]:
expect('coverage', coverage, 0.95)

The switchback interval contains the truth in about 95 of 100 seeds: it is honest, and wide. The test by customer is precise and wrong. In the project, the shared-stock test covers the truth in 0 of 100 seeds, and the switchback test in 96.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m04/when-units-affect-each-other/) covers randomisation by session, cluster randomisation and marketplaces, and has the knowledge checks that count toward your certificate. In the project, `sim run --scenario stock`, `switchback` and `session` are the three planted tests.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Difference-in-differences](https://learning.nextia-ai.com/courses/experiments/m05/difference-in-differences/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m04/when-units-affect-each-other/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.